In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import os

In [2]:
# Create outputs directory
os.makedirs("outputs", exist_ok=True)

In [3]:
# 1. Load Data
print("Loading clean dataset...")
df = pd.read_csv("../phase2/clean_model_dataset.csv")

Loading clean dataset...


In [4]:
# 2. Define Baseline Features
# As per README: Uses review metadata only. Example features: review length, helpful ratio, verified purchase.
baseline_features = [
    'star_rating',
    'verified_purchase',
    'review_length_words',
    'has_title',
    'title_word_count',
    'num_images_attached',
    'helpful_votes',
    'total_votes',
    'helpful_ratio',
    'days_since_purchase',
    'reviewer_review_count',
    'is_top_reviewer',
    'is_early_review'
]

In [5]:
target = 'is_fake_review'

In [6]:
X = df[baseline_features].copy()
y = df[target].copy()

In [7]:
# Fill any remaining NaNs with 0 or median (should be clean, but just in case)
X.fillna(X.median(), inplace=True)

,star_rating,verified_purchase,review_length_words,has_title,title_word_count,num_images_attached,helpful_votes,total_votes,helpful_ratio,days_since_purchase,reviewer_review_count,is_top_reviewer,is_early_review
0,5,1,34,1,3,0,18,19,0.947,7,38,1,0
1,4,1,62,1,11,0,1,2,0.500,7,6,0,0
2,5,1,104,0,0,0,2,2,1.000,21,38,1,0
3,5,1,92,1,10,0,32,41,0.780,30,42,1,0
4,3,1,110,1,6,5,8,12,0.667,15,44,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,3,1,24,1,6,0,2,3,0.667,5,19,0,0
99996,5,1,34,0,0,0,1,1,1.000,17,40,1,0
99997,3,0,34,1,10,2,69,73,0.945,12,49,1,0
99998,5,0,27,1,6,2,3,4,0.750,21,33,1,0


In [8]:
# 3. Train-Test Split
print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

Splitting data...
Train size: (80000, 13), Test size: (20000, 13)


In [9]:
# 4. Train Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
}

In [10]:
results = []

In [11]:
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"{name} Results:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    })
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Genuine (0)', 'Fake (1)'], 
                yticklabels=['Genuine (0)', 'Fake (1)'])
    plt.title(f'Confusion Matrix: {name} (Baseline)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    
    # Save plot
    filename = name.replace(" ", "_").lower()
    plt.savefig(f"outputs/cm_baseline_{filename}.png")
    plt.close()
    
    # Feature Importance for Random Forest
    if name == "Random Forest":
        importances = model.feature_importances_
        indices = np.argsort(importances)[::-1]
        
        plt.figure(figsize=(10, 6))
        sns.barplot(x=importances[indices], y=np.array(baseline_features)[indices], hue=np.array(baseline_features)[indices], palette="viridis", legend=False)
        plt.title('Feature Importances: Random Forest (Baseline)')
        plt.xlabel('Relative Importance')
        plt.tight_layout()
        plt.savefig(f"outputs/feature_importance_baseline_{filename}.png")
        plt.close()


Training Logistic Regression...


Logistic Regression Results:
Accuracy:  0.7441
Precision: 0.1643
Recall:    0.8161
F1-score:  0.2735

Training Random Forest...


Random Forest Results:
Accuracy:  0.9404
Precision: 0.1176
Recall:    0.0017
F1-score:  0.0033


In [12]:
# Save results
results_df = pd.DataFrame(results)
results_df.to_csv("outputs/baseline_results.csv", index=False)
print("\nSaved baseline evaluation results to outputs/baseline_results.csv")


Saved baseline evaluation results to outputs/baseline_results.csv
